# 10 - Zero-Shot Classification with Cloud LLMs (Groq + Gemini)

Classifies tweets using large cloud LLMs via OpenAI-compatible APIs: **Groq** (free tier) and **Google Gemini**.

### Models
| Key | Model | Parameters | Provider |
|-----|-------|------------|----------|
| `gpt_oss` | GPT-OSS 120B | 120B | Groq (free) |
| `llama33` | Llama 3.3 70B | 70B dense | Groq (free) |
| `qwen3` | Qwen3.6 27B | 27B | Groq (free) |
| `gemini_flash` | Gemini 2.5 Flash | proprietary | Google AI Studio |

### Datasets
- **Manchester** - reliable vs misinformation
- **Monkeypox** - reliable vs misinformation
- **PHEME** - not_rumour vs rumour

### Requirements
- `pip install openai python-dotenv`
- `GROQ_API_KEY` in `.env` (free key at console.groq.com) — for Groq models
- `GEMINI_API_KEY` in `.env` (free key at aistudio.google.com) — for Gemini

### Run order
Run once per combination: set `DATASET` + `MODEL`, then run all cells.
12 total runs (4 models × 3 datasets). Or use `scripts/run_all_cloud.py` to automate all of them.

## 0. Dataset & Model Selection

In [ ]:
# ============================================================
# CHANGE THESE TWO VARIABLES TO SWITCH RUNS
# DATASET:  'manchester' | 'monkeypox' | 'pheme'
# MODEL:    'gpt_oss'    | 'llama33'   | 'qwen3' | 'gemini_flash'
# ============================================================
DATASET = 'manchester'
MODEL   = 'gemini_flash'

# ── Dataset configs ───────────────────────────────────────────────────────────
DATASET_CONFIG = {
    'manchester': {
        'test':        '../../data/gold_standard/manchester_test.csv',
        'text_col':    'cleaned_tweet',
        'label_col':   'label',
        'label_map':   {'reliable': 0, 'misinformation': 1, 'unrelated': 2},
        'label_names': ['reliable', 'misinformation', 'unrelated'],
        'pos_label':   'misinformation',
        'topic':       'the 2017 Manchester Arena bombing',
        'classes': {
            'reliable':       'factually accurate, verified, or plausible news about the Manchester Arena bombing',
            'misinformation': 'false, unverified, or misleading claims about the event — rumours, conspiracy theories, or fabricated stories',
            'unrelated':      'the tweet is NOT about the Manchester Arena bombing at all — off-topic or irrelevant content',
        },
    },
    'monkeypox': {
        'test':        '../../data/gold_standard/monkeypox_test.csv',
        'text_col':    'cleaned_tweet',
        'label_col':   'label',
        'label_map':   {'reliable': 0, 'misinformation': 1, 'unrelated': 2},
        'label_names': ['reliable', 'misinformation', 'unrelated'],
        'pos_label':   'misinformation',
        'topic':       'the 2022 Monkeypox (Mpox) outbreak',
        'classes': {
            'reliable':       'factually accurate health information about Monkeypox symptoms, transmission, or treatment',
            'misinformation': 'false health claims, conspiracy theories, or misleading information about Monkeypox',
            'unrelated':      'the tweet is NOT genuinely about the Monkeypox outbreak — off-topic or irrelevant content',
        },
    },
    'pheme': {
        'test':        '../../data/gold_standard/pheme_test.csv',
        'text_col':    'cleaned_tweet',
        'label_col':   'label',
        'label_map':   {'not_rumour': 0, 'rumour': 1, 'unrelated': 2},
        'label_names': ['not_rumour', 'rumour', 'unrelated'],
        'pos_label':   'rumour',
        'topic':       'breaking news events (Charlie Hebdo attack 2015, Ferguson unrest 2014)',
        'classes': {
            'not_rumour': 'verified news, factual reporting, or confirmed information about the events',
            'rumour':     'unverified claims, speculation, or information that has not been confirmed by credible sources',
            'unrelated':  'the tweet is NOT about either tracked event (Charlie Hebdo / Ferguson) — off-topic or irrelevant content',
        },
    },
}

# ── Provider configs ──────────────────────────────────────────────────────────
PROVIDER_CONFIG = {
    'groq': {
        'base_url':    'https://api.groq.com/openai/v1',
        'api_key_env': 'GROQ_API_KEY',
    },
    'gemini': {
        # Google's OpenAI-compatible endpoint — same OpenAI SDK, different base_url
        'base_url':    'https://generativelanguage.googleapis.com/v1beta/openai/',
        'api_key_env': 'GEMINI_API_KEY',
    },
}

# ── Model configs ─────────────────────────────────────────────────────────────
# Groq models: https://console.groq.com/docs/models | Gemini: https://ai.google.dev/gemini-api/docs/models
MODEL_CONFIG = {
    'gpt_oss': {
        'provider':     'groq',
        'model_id':     'openai/gpt-oss-120b',
        'display_name': 'GPT-OSS 120B',
        'params_b':     120,
    },
    'llama33': {
        'provider':     'groq',
        'model_id':     'llama-3.3-70b-versatile',
        'display_name': 'Llama 3.3 70B',
        'params_b':     70,
    },
    'qwen3': {
        # qwen/qwen3-32b was removed from Groq (2026-07) — qwen3.6-27b is its successor
        'provider':     'groq',
        'model_id':     'qwen/qwen3.6-27b',
        'display_name': 'Qwen3.6 27B',
        'params_b':     27,
        # reasoning_effort 'none' fully disables qwen3.6 thinking
        'request_kwargs': {
            'extra_body': {'reasoning_effort': 'none'},
        },
    },
    'gemini_flash': {
        'provider':     'gemini',
        'model_id':     'gemini-2.5-flash',
        'display_name': 'Gemini 2.5 Flash',
        'params_b':     None,  # proprietary — parameter count not public
        # 2.5 Flash thinks by default; thinking burns tokens inside max_tokens
        # and can truncate the JSON answer — disable it for classification
        'request_kwargs': {
            'extra_body': {'extra_body': {'google': {'thinking_config': {'thinking_budget': 0}}}},
        },
    },
}

CFG       = DATASET_CONFIG[DATASET]
MODEL_CFG = MODEL_CONFIG[MODEL]
PROV_CFG  = PROVIDER_CONFIG[MODEL_CFG['provider']]

print(f"Dataset : {DATASET.upper()}")
print(f"Topic   : {CFG['topic']}")
print(f"Classes : {CFG['label_names']}")
print(f"Model   : {MODEL_CFG['display_name']}")
print(f"Provider: {MODEL_CFG['provider']}")
print(f"Model ID: {MODEL_CFG['model_id']}")

## 1. Imports & Setup

In [ ]:
import os
import re
import time
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
from openai import OpenAI, APIError, APITimeoutError, RateLimitError

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, accuracy_score,
)

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RESULTS_DIR = Path('../../results')
PREDS_DIR   = RESULTS_DIR / 'predictions'
FIGS_DIR    = RESULTS_DIR / 'figures'
for d in [PREDS_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Imports ready.')

## 2. API Setup

The key depends on the selected model's provider:
- **Groq** models — free key at **console.groq.com → API Keys**
- **Gemini** — free key at **aistudio.google.com → Get API key**

```bash
# Add to .env file:
GROQ_API_KEY=gsk_...
GEMINI_API_KEY=AIza...
```

In [ ]:
# Load .env if present (pip install python-dotenv)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

KEY_ENV = PROV_CFG['api_key_env']
API_KEY = os.environ.get(KEY_ENV, '')

if not API_KEY:
    raise EnvironmentError(
        f"{KEY_ENV} not set.\n"
        "Groq: free key at console.groq.com | Gemini: free key at aistudio.google.com\n"
        f"Add {KEY_ENV}=... to .env"
    )

client = OpenAI(
    base_url=PROV_CFG['base_url'],
    api_key=API_KEY,
)

API_TIMEOUT       = 60
MAX_RETRIES       = 5
RETRY_DELAY_BASE  = 10    # back-off if rate limited
MAX_TOKENS        = 400

print(f'{MODEL_CFG["provider"].capitalize()} client ready.')
print(f'Model  : {MODEL_CFG["display_name"]}')
print(f'Key    : {API_KEY[:12]}...{API_KEY[-4:]}')

## 3. Load Test Data

In [ ]:
df_test = pd.read_csv(CFG['test'])
df_test.dropna(subset=[CFG['text_col'], CFG['label_col']], inplace=True)
df_test[CFG['text_col']] = df_test[CFG['text_col']].astype(str)

LABEL_MAP = CFG['label_map']
ID2LABEL  = {v: k for k, v in LABEL_MAP.items()}

print(f'Test set: {len(df_test):,} samples')
print(f'\nLabel distribution:')
print(df_test[CFG['label_col']].value_counts())

## 4. Promptbook — Dataset-Specific Prompts

In [ ]:
SYSTEM_PROMPT = "You are an expert fact-checker and misinformation analyst specializing in social media content. Always respond with valid JSON only - no extra text before or after."

def build_user_prompt(tweet_text: str, cfg: dict) -> str:
    NL = chr(10)
    Q3 = chr(34) * 3
    class_lines   = NL.join(f'- "{name}": {desc}' for name, desc in cfg['classes'].items())
    label_options = " or ".join(f'"{name}"' for name in cfg['classes'])
    return f"""Your task: Classify the following tweet about {cfg['topic']}.

CLASSES:
{class_lines}

TWEET:
{Q3}{tweet_text}{Q3}

INSTRUCTIONS:
Think step-by-step before classifying. Consider:
1. Is the tweet actually about {cfg['topic']}, or is it off-topic/unrelated?
2. If on-topic: what specific claim does the tweet make?
3. Does it present verifiable facts, or unverified/emotional claims?
4. Are there signals of misinformation: conspiracy language, extreme emotion, lack of sources, implausible claims?
5. What is your final classification?

Respond in this exact JSON format (no extra text before or after):
{{
  "reasoning": "<your step-by-step reasoning in 2-4 sentences>",
  "label": {label_options},
  "confidence": <float between 0.0 and 1.0>
}}"""


# Preview
sample_tweet  = df_test[CFG['text_col']].iloc[0]
sample_prompt = build_user_prompt(sample_tweet, CFG)
print('=== PROMPT PREVIEW ===')
print(sample_prompt)
print(chr(10) + 'Tweet: ' + str(sample_tweet[:200]))

## 5. Inference with Retry Logic

In [ ]:
def call_api(user_prompt: str, max_retries: int = MAX_RETRIES) -> str:
    """
    Call the provider API (Groq / Gemini via OpenAI-compatible endpoint).
    Retries on timeout and rate-limit errors with exponential backoff.
    Returns empty string on total failure (never raises).
    """
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_CFG['model_id'],
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': user_prompt},
                ],
                temperature=0.0,
                max_tokens=MAX_TOKENS,
                timeout=API_TIMEOUT,
                **MODEL_CFG.get('request_kwargs', {}),
            )
            raw = response.choices[0].message.content or ''
            return raw

        except RateLimitError as e:
            import re as _re
            # Groq: retry_after_seconds | Gemini: retryDelay
            m = _re.search(r'(?:retry_after_seconds|retryDelay)\D*?([\d.]+)', str(e))
            wait = int(float(m.group(1))) + 5 if m else RETRY_DELAY_BASE * (2 ** attempt)
            print(f'  [RateLimit] waiting {wait}s before retry {attempt+1}/{max_retries}')
            time.sleep(wait)

        except APITimeoutError:
            print(f'  [Timeout] attempt {attempt+1}/{max_retries}')
            if attempt < max_retries - 1:
                time.sleep(RETRY_DELAY_BASE * (attempt + 1))

        except APIError as e:
            print(f'  [APIError] {e}')
            if attempt < max_retries - 1:
                time.sleep(3)

        except Exception as e:
            print(f'  [Unexpected] {e}')
            if attempt < max_retries - 1:
                time.sleep(3)

    return ''


def parse_response(response_text: str, cfg: dict) -> dict:
    """
    Parse JSON response. Handles <think> tags (Qwen3) and falls back to keyword matching.
    Returns dict with: label, confidence, reasoning, parse_error, parse_method.
    """
    if not response_text or not response_text.strip():
        return {
            'label': None, 'confidence': 0.5,
            'reasoning': 'Empty response from API',
            'parse_error': True, 'parse_method': 'empty',
        }

    # Primary: JSON parse
    try:
        # Strip <think>...</think> blocks (Qwen3 reasoning mode)
        clean = re.sub(r'<think>.*?</think>', '', response_text, flags=re.DOTALL).strip()
        clean = re.sub(r'```json\s*|```\s*', '', clean).strip()
        match = re.search(r'\{.*\}', clean, re.DOTALL)
        if match:
            data  = json.loads(match.group())
            label = str(data.get('label', '')).strip().lower()
            if label in cfg['label_map']:
                return {
                    'label':        label,
                    'confidence':   float(data.get('confidence', 0.5)),
                    'reasoning':    str(data.get('reasoning', '')),
                    'parse_error':  False,
                    'parse_method': 'json',
                }
    except (json.JSONDecodeError, ValueError, TypeError):
        pass

    # Fallback: keyword scan
    text_lower = response_text.lower()
    for label_name in sorted(cfg['label_names'], key=len, reverse=True):
        if label_name in text_lower:
            return {
                'label': label_name, 'confidence': 0.5,
                'reasoning': response_text[:300],
                'parse_error': True, 'parse_method': 'keyword_fallback',
            }

    return {
        'label': None, 'confidence': 0.0,
        'reasoning': response_text[:300],
        'parse_error': True, 'parse_method': 'default',
    }


def classify_tweet(tweet: str, cfg: dict) -> dict:
    prompt = build_user_prompt(tweet, cfg)
    raw    = call_api(prompt)
    return parse_response(raw, cfg)


print('Inference functions defined.')
print(f'Model: {MODEL_CFG["display_name"]} | Timeout: {API_TIMEOUT}s | Max retries: {MAX_RETRIES}')

## 6. Run Classification on Test Set

> Cloud API — rate limits apply. Checkpoint every 50 tweets for resume support.

In [ ]:
CHECKPOINT_EVERY = 50

raw_path        = PREDS_DIR / f'{DATASET}_{MODEL}_raw.csv'
checkpoint_path = PREDS_DIR / f'{DATASET}_{MODEL}_checkpoint.csv'

# Resume from checkpoint
done_indices = set()
results      = []

if checkpoint_path.exists():
    df_ckpt      = pd.read_csv(checkpoint_path)
    done_indices = set(df_ckpt['index'].tolist())
    results      = df_ckpt.to_dict('records')
    print(f'Checkpoint found: {len(done_indices):,} tweets already processed — resuming.')
else:
    print('No checkpoint — starting fresh.')

df_todo = df_test[~df_test.index.isin(done_indices)]
total   = len(df_test)

print(f'\nClassifying {len(df_todo):,} remaining tweets with {MODEL_CFG["display_name"]}...')
print(f'(Total: {total:,} | Already done: {len(done_indices):,})\n')

start_time    = time.time()
request_times = []

for n, (i, row) in enumerate(tqdm(df_todo.iterrows(), total=len(df_todo), desc='Classifying'), start=1):
    tweet      = row[CFG['text_col']]
    true_label = row[CFG['label_col']]

    t0     = time.time()
    result = classify_tweet(tweet, CFG)
    t1     = time.time()
    request_times.append(t1 - t0)

    results.append({
        'index':        i,
        'text':         tweet,
        'true_label':   true_label,
        'pred_label':   result['label'],
        'confidence':   result['confidence'],
        'reasoning':    result['reasoning'],
        'parse_error':  result['parse_error'],
        'parse_method': result['parse_method'],
    })

    processed_total = len(done_indices) + n
    avg_time        = sum(request_times) / len(request_times)
    remaining_n     = total - processed_total
    eta_sec         = avg_time * remaining_n

    if n % 10 == 0 or n == len(df_todo):
        print(
            f'  {processed_total:,}/{total:,} ({processed_total/total*100:.1f}%) | '
            f'avg {avg_time:.1f}s/tweet | ETA ~{eta_sec/60:.1f} min'
        )

    if n % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(checkpoint_path, index=False)
        print(f'  [Checkpoint saved: {len(results):,} rows]')

elapsed    = time.time() - start_time
df_results = pd.DataFrame(results)
null_count = df_results['pred_label'].isnull().sum()

df_results.to_csv(raw_path, index=False)
if checkpoint_path.exists():
    checkpoint_path.unlink()

print(f'\nDone! {len(df_results):,} classified in {elapsed/60:.1f} min')
print(f'Null / parse failures: {null_count} ({null_count/len(df_results)*100:.1f}%)')
print(f'\nParse method breakdown:')
for method, count in df_results['parse_method'].value_counts().items():
    print(f'  {method:<22}: {count:,} ({count/len(df_results)*100:.1f}%)')

## 7. Handle Errors & Finalize Predictions

In [ ]:
null_mask = df_results['pred_label'].isnull()
print(f'Null predictions: {null_mask.sum()} / {len(df_results)} ({null_mask.sum()/len(df_results)*100:.1f}%)')

if null_mask.sum() > 0:
    print('\nSample failed predictions:')
    print(df_results[null_mask][['text', 'reasoning', 'parse_method']].head(3).to_string())

majority_class = df_results['true_label'].mode()[0]
df_results['pred_label_final'] = df_results['pred_label'].fillna(majority_class)
df_results.loc[null_mask, 'confidence'] = df_results.loc[null_mask, 'confidence'].replace(0.0, 0.5)

print(f'\nFilled {null_mask.sum()} nulls with majority class: "{majority_class}"')
print(f'\nPrediction distribution:')
print(df_results['pred_label_final'].value_counts())

## 8. Evaluation Metrics

In [ ]:
y_true = df_results['true_label'].map(LABEL_MAP).values
y_pred = df_results['pred_label_final'].map(LABEL_MAP).values

print(f'\n{"="*60}')
print(f' {DATASET.upper()} — {MODEL_CFG["display_name"]} Zero-Shot Results')
print(f'{"="*60}')
print(classification_report(y_true, y_pred, target_names=CFG['label_names'], digits=4))

pos_label_int = LABEL_MAP[CFG['pos_label']]
parse_counts  = df_results['parse_method'].value_counts().to_dict()

metrics = {
    'dataset':              DATASET,
    'model':                MODEL,
    'model_display':        MODEL_CFG['display_name'],
    'params_b':             MODEL_CFG['params_b'],
    'provider':             MODEL_CFG['provider'],
    'model_id':             MODEL_CFG['model_id'],
    # test_ prefix for compatibility with notebook 09
    'test_accuracy':        accuracy_score(y_true, y_pred),
    'test_f1_macro':        f1_score(y_true, y_pred, average='macro'),
    'test_f1_weighted':     f1_score(y_true, y_pred, average='weighted'),
    'test_precision':       precision_score(y_true, y_pred, average='macro', zero_division=0),
    'test_recall':          recall_score(y_true, y_pred, average='macro', zero_division=0),
    f'test_f1_{CFG["pos_label"]}': f1_score(y_true, y_pred, labels=[pos_label_int], average='macro', zero_division=0),
    'null_predictions':     int(null_mask.sum()),
    'parse_errors':         int(df_results['parse_error'].sum()),
    'parse_json':           parse_counts.get('json', 0),
    'parse_keyword_fallback': parse_counts.get('keyword_fallback', 0),
    'parse_empty':          parse_counts.get('empty', 0),
    'parse_default':        parse_counts.get('default', 0),
}

print('--- Summary ---')
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k:<40}: {v:.4f}')
    else:
        print(f'  {k:<40}: {v}')

## 9. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CFG['label_names'], yticklabels=CFG['label_names'], ax=axes[0])
axes[0].set_title(f'{DATASET.upper()} — {MODEL_CFG["display_name"]} (Counts)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=CFG['label_names'], yticklabels=CFG['label_names'], ax=axes[1])
axes[1].set_title(f'{DATASET.upper()} — {MODEL_CFG["display_name"]} (Normalized)', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
fig_path = FIGS_DIR / f'{DATASET}_{MODEL}_cm.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## 10. Confidence Distribution

In [ ]:
df_results['correct'] = (df_results['true_label'] == df_results['pred_label_final'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

correct_conf   = df_results[df_results['correct']]['confidence']
incorrect_conf = df_results[~df_results['correct']]['confidence']

axes[0].hist(correct_conf,   bins=20, alpha=0.6, color='steelblue', label=f'Correct (n={len(correct_conf)})')
axes[0].hist(incorrect_conf, bins=20, alpha=0.6, color='crimson',   label=f'Incorrect (n={len(incorrect_conf)})')
axes[0].set_xlabel('Confidence'); axes[0].set_ylabel('Count')
axes[0].set_title(f'{DATASET.upper()} — Confidence by Outcome', fontweight='bold')
axes[0].legend(); axes[0].axvline(0.5, color='black', linestyle='--', alpha=0.5)

for label_name in CFG['label_names']:
    subset = df_results[df_results['pred_label_final'] == label_name]['confidence']
    axes[1].hist(subset, bins=20, alpha=0.6, label=f'{label_name} (n={len(subset)})')
axes[1].set_xlabel('Confidence'); axes[1].set_ylabel('Count')
axes[1].set_title(f'{DATASET.upper()} — Confidence by Predicted Label', fontweight='bold')
axes[1].legend()

plt.tight_layout()
fig_path = FIGS_DIR / f'{DATASET}_{MODEL}_confidence.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Mean confidence — correct: {correct_conf.mean():.3f} | incorrect: {incorrect_conf.mean():.3f}')

## 11. Save Predictions & Summary

In [ ]:
df_results['true_label_int'] = df_results['true_label'].map(LABEL_MAP)
df_results['pred_label_int'] = df_results['pred_label_final'].map(LABEL_MAP)

pred_path    = PREDS_DIR / f'{DATASET}_{MODEL}_test_predictions.csv'
summary_path = PREDS_DIR / f'{DATASET}_{MODEL}_summary.csv'

df_results.to_csv(pred_path, index=False)
pd.DataFrame([metrics]).to_csv(summary_path, index=False)

print(f'Predictions saved : {pred_path}')
print(f'Summary saved     : {summary_path}')
print(f'\nColumns: {list(df_results.columns)}')

print(f'\n=== FINAL RESULTS ===')
print(f'  Dataset  : {DATASET.upper()}')
print(f'  Model    : {MODEL_CFG["display_name"]}')
print(f'  Accuracy : {metrics["test_accuracy"]:.4f}')
print(f'  F1 Macro : {metrics["test_f1_macro"]:.4f}')
print(f'  Precision: {metrics["test_precision"]:.4f}')
print(f'  Recall   : {metrics["test_recall"]:.4f}')

## 12. Results Summary Table

In [ ]:
pos_key = f'test_f1_{CFG["pos_label"]}'

metrics_display = [
    ['Metric',                  'Value'],
    ['Accuracy',                f"{metrics['test_accuracy']:.4f}"],
    ['F1 Macro',                f"{metrics['test_f1_macro']:.4f}"],
    ['F1 Weighted',             f"{metrics['test_f1_weighted']:.4f}"],
    ['Precision (Macro)',       f"{metrics['test_precision']:.4f}"],
    ['Recall (Macro)',          f"{metrics['test_recall']:.4f}"],
    [f'F1 ({CFG["pos_label"]})',f"{metrics[pos_key]:.4f}"],
    ['Parse: JSON',             str(metrics['parse_json'])],
    ['Parse: keyword fallback', str(metrics['parse_keyword_fallback'])],
    ['Parse: empty/default',    str(metrics['parse_empty'] + metrics['parse_default'])],
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.axis('off')
table = ax.table(
    cellText=metrics_display[1:],
    colLabels=metrics_display[0],
    cellLoc='center', loc='center',
    bbox=[0.15, 0, 0.7, 1]
)
table.auto_set_font_size(False)
table.set_fontsize(10)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2980b9')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#eaf4fb')

ax.set_title(
    f'{DATASET.upper()} — {MODEL_CFG["display_name"]} Zero-Shot Results',
    fontsize=13, fontweight='bold', pad=20
)
plt.tight_layout()
fig_path = FIGS_DIR / f'{DATASET}_{MODEL}_summary_table.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')
print(f'\nNext: change DATASET / MODEL at the top and re-run all cells.')